# DS203 from zero to exam practice

Run cells in order with **Shift+Enter**. Read the explanation, predict the output, run it, then use the practice prompt to write a variation yourself. Restart the kernel and run all if a variable is unexpectedly missing or stale.

The first Python cell needs only Python. Later sections use NumPy, pandas, Matplotlib, scipy, scikit-learn and statsmodels, as in the course. The examples generate their data locally or use datasets bundled with scikit-learn. The cricket example uses the supplied ZIP when it is beside this notebook and has a small built-in fallback. If your kernel needs packages and supports installation, run `%pip install numpy pandas matplotlib scipy scikit-learn statsmodels` in its own cell, then restart the kernel. Use the companion **DS203_Exam_Review.md** for written recall, study timing and a timed mock.

## 1. Python and text

A variable names a value. A list holds items. A loop visits each item. An `if` selects a branch. A function takes inputs and returns a result. The lines inside each block are indented. A dictionary maps a key to a value. `get(key, 0)` gives zero for a new key.

Example goal: count words in a sentence. Split the text, look up the old count, then add one. The comprehension at the end constructs a list by visiting each item and retaining the items that pass a condition.

**Practice.** Rewrite the word counter as a function. Change the text to `"PCA, PCA! kmeans"` and remove punctuation before counting.

In [ ]:
import string

text = "Bayes, Bayes and KNN!"
counts = {}
for word in text.lower().split():
    cleaned = word.strip(string.punctuation)
    if cleaned:
        counts[cleaned] = counts.get(cleaned, 0) + 1
print(counts)

runs = [0, 12, 50, 80]
print(runs[0], runs[-1], runs[1:3])
print([score for score in runs if score >= 50])
print(sum(runs) / len(runs))

other_name = runs
copied_runs = runs.copy()
other_name.append(25)
print("Shared list:", runs, "Separate copy:", copied_runs)

## 2. NumPy, pandas, plots, and exploratory data analysis

An array turns operations over numbers into a single expression. The comparison `scores >= 50` produces True/False values. Selecting with that mask returns the matching scores. Averaging the mask counts the fraction that match. In a two-dimensional array, use `axis=0` to take a statistic down the rows for each column.

A DataFrame is a labeled table. Load a dataset with `pd.read_csv("file.csv")`, optionally with `sep=";"` or `na_values=["AA"]` if the file needs it. Inspect rows, types and missingness. Use `loc` for labels and Boolean conditions, `iloc` for positions, and `groupby` for per-group statistics. When computing a proportion, decide whether missing observations belong in its denominator.

A histogram shows distribution, a box plot compares groups, and a scatter plot shows how two numeric variables move together. Describe a result using a number and a comparison. A higher mean and a lower standard deviation answer different questions.

**Practice.** Add a column showing the fraction of scores equal to zero. Sort by that fraction. Then change the cutoff to 30 and recompute the threshold fraction.

In [ ]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

scores = np.array([0, 12, 50, 80])
print("Scores >= 50:", scores[scores >= 50], "Fraction:", (scores >= 50).mean())
print("Column means:", np.array([[1, 10], [3, 20], [5, 30]]).mean(axis=0))

archive_path = Path("batsmen (1).zip")
players = {"Dhoni": [0, 31, 60], "Kohli": [5, 50, 91], "Sachin": [8, 17, 72]}
if archive_path.exists():
    with ZipFile(archive_path) as archive:
        players = {
            player: np.loadtxt(BytesIO(archive.read(f"{player.lower()}.txt")))
            for player in players
        }

batting = pd.DataFrame([
    {"player": player, "runs": score}
    for player, player_scores in players.items()
    for score in player_scores
])
print(batting.head())
print(batting.isna().sum())
print(batting.loc[batting["runs"] >= 100, ["player", "runs"]].head())
observed = batting.dropna(subset=["runs"]).copy()
observed["fifty"] = observed["runs"] >= 50
print(observed.groupby("player").agg(
    n=("runs", "size"), mean_runs=("runs", "mean"),
    p50=("fifty", "mean"), sd_runs=("runs", "std")
).round(3))

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(batting["runs"].dropna(), bins=15)
ax[0].set(xlabel="Runs", ylabel="Innings")
batting.boxplot(column="runs", by="player", ax=ax[1])
ax[1].set(xlabel="Player", ylabel="Runs")
fig.suptitle("")
fig.tight_layout()
plt.show()

In [ ]:
# Read and clean an example CSV without needing an external file.
from io import StringIO

csv_text = "group;hours;score\nA;1;40\nA;2;55\nB;3;AA\nB;5;85\n"
survey = pd.read_csv(StringIO(csv_text), sep=";", na_values=["AA"])
survey.info()
print(survey.describe())
print(survey.iloc[:2, :2])
print(survey[["hours", "score"]].corr())
print(pd.crosstab(survey["group"], survey["score"].isna()))

complete = survey.dropna(subset=["score"])
complete.plot.scatter(x="hours", y="score")
plt.show()

## 3. Statistics, simulation, and the central limit theorem

A population parameter describes the underlying distribution. A sample statistic, such as the sample mean, is calculated from observations and can vary across samples. The standard deviation of observations and the standard error of their mean have different denominators conceptually. For independent observations with population SD σ, the mean has SD σ/√n. Use `ddof=1` for the ordinary sample variance or SD estimate in NumPy.

Simulation approximates a probability by the fraction of repetitions in which an event happens. Here each row is one simulated experiment. The exponential distribution is right-skewed. Compare the distribution of raw observations with the distribution of averages. The CLT says the centered and standardized mean approaches a normal distribution under i.i.d. finite-variance assumptions.

**Practice.** Increase n from 25 to 100. Predict what happens to the SD of the sample means before running the code. For Pareto shape 1.5 explain why the classical CLT cannot be used.

In [ ]:
from scipy.stats import pareto

rng = np.random.default_rng(203)
n = 25
simulated = rng.exponential(scale=2, size=(5000, n))
sample_means = simulated.mean(axis=1)

print("Mean of means:", round(sample_means.mean(), 3))
print("Simulated SE:", round(sample_means.std(ddof=1), 3))
print("Theoretical SE:", 2 / np.sqrt(n))
print("Estimated P(sample mean > 2.5):", (sample_means > 2.5).mean())
for shape in [1.5, 2.5, 4]:
    print("Pareto shape, mean, variance:", shape, *pareto.stats(shape, moments="mv"))

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(simulated[:, 0], bins=35)
ax[0].set_title("One draw per experiment")
ax[1].hist(sample_means, bins=35)
ax[1].set_title(f"Average of {n} draws per experiment")
fig.tight_layout()
plt.show()

## 4. Bayes from first principles

A prior gives probabilities before seeing data. A likelihood asks how probable the observed data are under each possible parameter or class. Multiply them and divide by the sum (or integral) to get the posterior.

If P(disease)=0.10, P(positive | disease)=0.80 and P(positive | healthy)=0.20, imagine 100 people. About 8 sick people and 18 healthy people test positive. So P(disease | positive)=8/26≈0.308. Under equal error costs, predict healthy given a positive. When costs differ, choose the prediction with the lower expected cost.

For a coin with head probability θ, the likelihood of a sequence of h heads and t tails is θ^h(1−θ)^t. On an evenly spaced grid with equal initial weights, posterior probability masses are proportional to the likelihood. The posterior mean is the weighted average of θ. A uniform Beta(1,1) prior and 7 heads in 10 tosses give Beta(8,4) and mean 8/12.

**Practice.** Change the grid prior to only support 0.4–0.6. Explain whether any number of heads can move the posterior outside that interval.

In [ ]:
theta = (np.arange(1000) + 0.5) / 1000
heads, tails = 7, 3
prior = np.ones_like(theta)
prior = prior / prior.sum()
likelihood = theta**heads * (1 - theta)**tails
posterior = prior * likelihood
posterior = posterior / posterior.sum()

print("Posterior mean:", theta @ posterior)
plt.plot(theta, prior, label="Prior mass")
plt.plot(theta, posterior, label="Posterior mass")
plt.xlabel("Probability of heads")
plt.legend()
plt.show()

## 5. Linear regression by hand and in code

You have known inputs x and targets y. Fit an equation for new x values. OLS chooses intercept and slope to minimize the sum of squared residuals. Residual = actual − predicted. With x = [1,2,3] and y = [2,3,7], x̄ = 2 and ȳ = 4. The sum of centered products is 5 and the sum of squared centered x values is 2, so slope = 2.5 and intercept = 4 − 2.5×2 = −1. Plug x into the line to get predictions [1.5,4,6.5]. SSE = 0.5² + (−1)² + 0.5² = 1.5. R² = 1 − SSE/SST ≈ 0.893 since SST = 14.

In scikit-learn, X needs shape (rows, features) and y is usually a vector. `x.reshape(-1, 1)` makes one feature column. With a DataFrame, `df[["hours"]]` makes a table while `df["hours"]` makes a Series.

Use statsmodels for summaries and inference. In its formula syntax, `C(group)` treats a column as categorical and `hours * C(group)` means both main effects and their interaction. Explain interaction as a change in the effect of one predictor depending on the other. A 95% confidence interval for a coefficient quantifies uncertainty in that coefficient. A residual plot is useful for checking nonlinear patterns and nonconstant spread.

**Practice.** For ŷ = 2 + 3x + 5G − 2xG, write the equation for each group and predict at x=4.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

x = np.array([1, 2, 3])
y = np.array([2, 3, 7])
xc, yc = x - x.mean(), y - y.mean()
slope = (xc @ yc) / (xc @ xc)
intercept = y.mean() - slope * x.mean()

model = LinearRegression().fit(x.reshape(-1, 1), y)
fitted = model.predict(x.reshape(-1, 1))
print("Manual intercept and slope:", intercept, slope)
print("Model intercept and slope:", model.intercept_, model.coef_[0])
print("Residuals:", y - fitted)
print("Training MSE:", mean_squared_error(y, fitted))
print("Training R²:", r2_score(y, fitted))

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import probplot
from statsmodels.stats.outliers_influence import variance_inflation_factor

rng = np.random.default_rng(23)
hours = rng.uniform(0, 10, 160)
group = rng.choice(["A", "B"], size=160)
is_b = group == "B"
scores = 2 + 3 * hours + 5 * is_b - 2 * hours * is_b + rng.normal(0, 2, 160)
study = pd.DataFrame({"hours": hours, "group": group, "score": scores})
study_model = smf.ols("score ~ hours * C(group)", data=study).fit()
additive_model = smf.ols("score ~ hours + C(group)", data=study).fit()
print(study_model.params.round(3))
print(study_model.conf_int().round(3))
print("Adjusted R²:", round(study_model.rsquared_adj, 3))
print("Interaction F test (F, p, df difference):", study_model.compare_f_test(additive_model))
other_hours = hours + rng.normal(0, 0.4, len(hours))
vif_design = np.column_stack([np.ones(len(hours)), hours, other_hours])
print("VIF for hours:", round(variance_inflation_factor(vif_design, 1), 2))

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].scatter(study_model.fittedvalues, study_model.resid, alpha=0.5)
ax[0].axhline(0, color="black")
ax[0].set(xlabel="Fitted score", ylabel="Residual")
probplot(study_model.resid, plot=ax[1])
fig.tight_layout()
plt.show()

## 6. Regularization, cross-validation and feature pipelines

Many features or strong collinearity can make estimated slopes unstable. Ridge adds a squared coefficient penalty and shrinks slopes. Lasso adds an absolute coefficient penalty and can yield zeros. Elastic net combines the two. For polynomial regression, include powers as features. More terms can fit the training points closely while predicting poorly on fresh points.

Tradeoff example: if two distance features are measured in rupees and years, raw numbers may cause rupees to dominate. If inputs go into ridge or lasso, large units can also affect how strongly a feature is penalized. A standard scaler learns feature means and SDs from training rows and applies them later.

**Train/test/CV sequence.** Split off a test set first. On the training set, try each hyperparameter using the same K folds. Within every fold, fit preprocessing and the estimator on its training rows. Score the remaining fold. Pick the best mean score, refit on all training rows, then score the held-out test set once. Below, a mixed-data pipeline applies median imputation and scaling to numeric features, and mode imputation and one-hot encoding to category features.

**Practice.** Add another alpha to the grid. Change the model to lasso and keep the same consistent names or update both names. Explain why fitting the scaler before CV leaks information even when a test set was split off first.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

mixed = study.copy()
mixed.loc[mixed.index[::13], "hours"] = np.nan
mixed.loc[mixed.index[::17], "group"] = np.nan

X = mixed[["hours", "group"]]
y = mixed["score"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
preprocessing = ColumnTransformer([
    ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["hours"]),
    ("category", make_pipeline(SimpleImputer(strategy="most_frequent"),
                               OneHotEncoder(handle_unknown="ignore")), ["group"])
])
ridge_pipeline = Pipeline([("prep", preprocessing), ("ridge", Ridge())])
folds = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_search = GridSearchCV(
    ridge_pipeline, {"ridge__alpha": [0.01, 0.1, 1, 10]},
    scoring="neg_mean_squared_error", cv=folds
)
ridge_search.fit(X_train, y_train)
predicted = ridge_search.predict(X_test)
print("Best alpha:", ridge_search.best_params_)
print("CV MSE:", -ridge_search.best_score_)
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, predicted)))
print("Test R²:", r2_score(y_test, predicted))

In [ ]:
# A manual version of the same CV idea on a numerical dataset.
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import PolynomialFeatures

X_numeric, y_numeric = load_diabetes(return_X_y=True, scaled=False)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_numeric, y_numeric, test_size=0.25, random_state=42
)
mean_fold_error = {}
for alpha in [0.01, 0.1, 1, 10]:
    fold_errors = []
    for train_rows, validation_rows in folds.split(X_tr):
        fold_model = make_pipeline(PolynomialFeatures(2, include_bias=False),
                                   StandardScaler(), Ridge(alpha=alpha))
        fold_model.fit(X_tr[train_rows], y_tr[train_rows])
        fold_predictions = fold_model.predict(X_tr[validation_rows])
        fold_errors.append(mean_squared_error(y_tr[validation_rows], fold_predictions))
    mean_fold_error[alpha] = np.mean(fold_errors)

best_alpha = min(mean_fold_error, key=mean_fold_error.get)
final_model = make_pipeline(PolynomialFeatures(2, include_bias=False),
                            StandardScaler(), Ridge(alpha=best_alpha))
final_model.fit(X_tr, y_tr)
print("Mean fold MSE by alpha:", mean_fold_error)
print("Final test MSE:", mean_squared_error(y_te, final_model.predict(X_te)))

## 7. Classification, LDA, KNN and SVM

Classification predicts a discrete label. Many models also provide a score or probability. A threshold turns scores into labels.

**LDA** is a Bayes classifier that models each class as a Gaussian with a class-specific mean and shared covariance. In one dimension, score = xμ/σ² − μ²/(2σ²) + log(prior). At means 2 and 6, equal priors and variance 4, the scores tie at x=4. With unequal priors, the boundary shifts. Pick the class with the greater score.

**KNN** classifies by a vote among the K nearest training points. KNN regression averages their target values. For points 1:A, 3:A, 5:B, 8:B and x=4.2, nearest first: 5,3,1,8. 1-NN predicts B. 3-NN predicts A. A small K follows fine detail and noise. Choose K using validation.

**SVM** picks a separating surface using the margin. A soft margin allows violations and C controls how much to penalize them. An RBF kernel gives curved boundaries. Larger gamma makes the influence of each point more local. Logistic regression instead applies the sigmoid to a weighted sum of the features.

**Metrics.** If rows are true labels and columns are predictions in [0,1] order, the entries are TN,FP on the first row and FN,TP on the second. Precision is the fraction of predicted positives that are truly positive. Recall is the fraction of actual positives caught. F1 balances precision and recall. ROC uses false positive rate and recall across thresholds, so pass probabilities or decision scores to ROC rather than ordinary predictions.

**Practice.** For TN=80, FP=10, FN=5, TP=5 calculate accuracy, precision, recall and F1. Explain how a classifier that always predicts the majority class could earn high accuracy. Compare the four models below on the same split.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix, f1_score
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.svm import SVC

breast_cancer = load_breast_cancer(as_frame=True)
X = breast_cancer.data
y = (breast_cancer.target == 0).astype(int)  # 1 means malignant in this notebook.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

classifiers = {
    "LDA": make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()),
    "Logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN": GridSearchCV(
        make_pipeline(StandardScaler(), KNeighborsClassifier()),
        {"kneighborsclassifier__n_neighbors": [3, 5, 9]}, cv=5, scoring="f1"
    ),
    "SVM": GridSearchCV(
        make_pipeline(StandardScaler(), SVC(kernel="rbf")),
        {"svc__C": [0.1, 1, 10], "svc__gamma": ["scale", 0.01, 0.1]},
        cv=5, scoring="f1"
    )
}
fig, ax = plt.subplots(figsize=(5, 4))
for name, classifier in classifiers.items():
    classifier.fit(X_train, y_train)
    predicted = classifier.predict(X_test)
    print(name, "test F1:", round(f1_score(y_test, predicted), 3))
    print(confusion_matrix(y_test, predicted, labels=[0, 1]))
    RocCurveDisplay.from_estimator(classifier, X_test, y_test, name=name, ax=ax)
print("KNN CV choice:", classifiers["KNN"].best_params_)
print("SVM CV choice:", classifiers["SVM"].best_params_)
fig.tight_layout()
plt.show()

# KNN regression averages nearby numeric targets.
neighbor_regressor = KNeighborsRegressor(n_neighbors=2)
neighbor_regressor.fit([[1], [3], [5], [8]], [10, 30, 50, 80])
print("2-NN regression at x=4.2:", neighbor_regressor.predict([[4.2]])[0])

## 8. K-means and PCA

In clustering there is no known target. K-means alternates between assigning points to their closest centroid and replacing centroids with cluster means. For data [1,2,7,10] and centroids [1,7], assignment gives {1,2} and {7,10}. The new means are [1.5,8.5]. Choose K using domain context, elbow or silhouette scores. Scaling affects distance.

PCA produces new numerical features. Center the data, choose the direction that preserves the most variation, and project the rows onto that direction. In matrix terms, compute the covariance matrix, find its eigenvectors, order them from largest eigenvalue down, and multiply centered data by the chosen eigenvectors. For [(1,1),(2,2),(3,3)] the mean is (2,2), covariance is [[1,1],[1,1]], variances along the PCs are 2 and 0, and the first direction is proportional to (1,1). Either sign is valid. Variance explained by component j is its eigenvalue divided by the sum of all eigenvalues.

**Practice.** Do the first Lloyd assignment and update with starting centroids 1 and 2 instead. With PCA eigenvalues 5,3,2, decide how many components explain at least 75%. Explain why scaling a feature before PCA can change the principal directions.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

points = np.array([1., 2., 7., 10.])
centers = np.array([1., 7.])
squared_distances = (points[:, None] - centers[None, :])**2
labels = squared_distances.argmin(axis=1)
print("One Lloyd update:", [points[labels == j].mean() for j in range(len(centers))])

toy = np.array([[1., 1.], [2., 2.], [3., 3.]])
centered = toy - toy.mean(axis=0)
eigenvalues, eigenvectors = np.linalg.eigh(np.cov(centered, rowvar=False))
print("Covariance:", np.cov(centered, rowvar=False))
print("Eigenvalues from largest:", eigenvalues[::-1])
print("Projection onto PC1:", centered @ eigenvectors[:, -1])

features, _ = make_blobs(n_samples=240, n_features=3, centers=3, random_state=42)
scaled_features = StandardScaler().fit_transform(features)
for k in [2, 3, 4, 5]:
    clusters = KMeans(n_clusters=k, n_init=10, random_state=42).fit(scaled_features)
    quality = silhouette_score(scaled_features, clusters.labels_)
    print(f"K={k} inertia={clusters.inertia_:.1f} silhouette={quality:.3f}")

pca = PCA(n_components=2)
coordinates = pca.fit_transform(scaled_features)
print("Explained variance ratios:", pca.explained_variance_ratio_)
clusters = KMeans(n_clusters=3, n_init=10, random_state=42).fit(scaled_features)

plt.scatter(coordinates[:, 0], coordinates[:, 1], c=clusters.labels_, cmap="viridis")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("K-means labels viewed in two principal components")
plt.show()

## 9. Recall before opening the mock

1. Write a function that counts words, then use NumPy to compute a threshold fraction and pandas to compute it by group.
2. Explain where the test data enters a machine-learning pipeline and how CV differs from a test set.
3. Compute one OLS slope, one Bayes posterior, one confusion matrix metric, one KNN prediction, and one K-means update on paper.
4. Name a plot or diagnostic for each of these: skew, outliers, a mixed-group correlation, regression residual variance, discrimination across cutoffs.
5. Open **DS203_Exam_Review.md** at *Closed-note mock exam*. Use the answer key after you finish.